In [19]:
%matplotlib notebook
import numpy as np
import cvxopt
import cvxopt.glpk

In [20]:
cvxopt.glpk.ilp?

In [21]:
## To help create the huge matrix

def unravel(i,j,k,size=3):
    n2 = size*size
    assert(i>=0 and i<n2)
    assert(j>=0 and i<n2)
    assert(k>=0 and i<n2)
    
    return(k+ j*n2+ i*n2*n2)


### not used
def ravel(l,size=3):
    n2=size*size
    assert (l>=0 and l < n2*n2*n2)
    i = l // (n2*n2)
    j = (l % (n2*n2)) // n2
    k = l - i*n2*n2 - j*n2
    return ((i,j,k))

In [22]:
## create constraint matrix

def createMatrix(size=3):
    n2=size*size
    A=np.zeros((4*n2*n2,n2*n2*n2))
    return (A)

A=createMatrix()
A.shape

(324, 729)

In [23]:

## modifies a global constraint matrix
def sudokuConstraints(size=3):
    global A

    n2 = size*size
    ## line constraints: only one number per line

    r = 0 ## row index 

    for k in range(n2): ## for all numbers
        for j in range(n2): ## for all columns
            for i in range(n2): 
                A[r,unravel(i,j,k)] = 1 ## only one number k on line i
            r += 1
        
    ## column constraints: only one number per column
    for k in range(n2):
        for i in range(n2):
            for j in range(n2):
                A[r,unravel(i,j,k)] = 1
            r += 1
        
    # unicity constraints
    for i in range(n2): ## for all rows
        for j in range(n2): ## for all columns
            for k in range(n2):
                A[r, unravel(i,j,k)] = 1 ## only one number in each position
            r += 1
        
    # sub-square constraints
    for k in range(n2): ## for all numbers
        for i in range(size): ## for all row square
            for j in range(size): ## for all columns squares
                for u in range(size): ## all the lines in the subsquare
                    for v in range(size): ## all the columns in the subsquare
                        A[r, unravel(i*size+u,j*size+v,k)]=1
                r += 1
            
    print("Total number of constraints=",r)
    return;
    
    
def testA(A,order=3):
    n2 = order*order
    c = 4*(order*order)**2
    for n in range(c):
        if (np.sum(A[n,])!=n2):
            print("error on line", n)
            break
    print("All constraints OK")
    return

sudokuConstraints()
testA(A)

Total number of constraints= 324
All constraints OK


In [24]:
## add the fixed numbers constraints
Keasy = np.array([
[0,8,0, 9,0,1, 0,5,0],
[0,0,2, 6,8,7, 3,0,0],
[0,0,3, 0,0,0, 6,0,0],
[3,9,0, 0,0,0, 0,6,5],
[6,0,0, 4,7,5, 0,0,3],
[5,7,0, 0,0,0, 0,8,4],
[0,0,9, 0,0,0, 8,0,0],
[0,0,5, 1,2,4, 9,0,0],
[0,4,0, 8,0,3, 0,2,0]])

Khard = np.array(
    [
        [7,0,0, 0,0,0, 4,0,0],
        [0,2,0, 0,7,0, 0,8,0],
        [0,0,3, 0,0,8, 0,0,9],     
        [0,0,0, 5,0,0, 3,0,0],
        [0,6,0, 0,2,0, 0,9,0],
        [0,0,1, 0,0,7, 0,0,6],
        [0,0,0, 3,0,0, 9,0,0],
        [0,3,0, 0,4,0, 0,6,0],
        [0,0,9, 0,0,1, 0,0,5]
    ]
)

def fillfixed(K,order=3):
    global A
    n2=order*order
    for i in range(n2):
        for j in range(n2):
            k = K[i,j]
            if (k>0):
                newrow=np.zeros(n2*n2*n2)
                newrow[unravel(i,j,k-1)]=1
                A=np.vstack((A,newrow))
            
## careful, modifies A
fillfixed(Khard)
            
print("A.shape=", A.shape)

A.shape= (348, 729)


In [26]:
## solving
from cvxopt import matrix

def solveSudoku(A):
    b=matrix(np.ones(A.shape[0])) ## set partition
    c=matrix(np.zeros(A.shape[1])) ## zero cost
    G=matrix(np.zeros(A.shape))
    h=matrix(np.zeros(A.shape[0]))
    binary=np.array(range(A.shape[1]))
    I=set(binary)
    B=set(binary)
    Aeq = matrix(A)
    (status, solution) = cvxopt.glpk.ilp(c=c,G=G,h=h,
                                         A=Aeq,b=b,
                                         B=set(range(A.shape[1])))
    print(status)
    return(status,solution)


In [27]:
order = 3
n2 = order*order

## print solution
def printsol(sol):
    sep3="+-----+-----+-----+"
    for i in range(n2):
        if (i%3 == 0):
            print(sep3)
        print("|",end='')
        for j in range(n2):
            for k in range(n2):
                if (sol[unravel(i,j,k)]==1):
                    print(k+1,end='')
            if (j%3 ==2):
                print("|",end='')
            else:
                print(" ",end='')
        print("")
    print(sep3)
        
        
(status,solution) = solveSudoku(A)        
printsol(solution)
          

optimal
+-----+-----+-----+
|7 9 8|6 3 5|4 2 1|
|1 2 6|9 7 4|5 8 3|
|4 5 3|2 1 8|6 7 9|
+-----+-----+-----+
|9 7 2|5 8 6|3 1 4|
|5 6 4|1 2 3|8 9 7|
|3 8 1|4 9 7|2 5 6|
+-----+-----+-----+
|6 1 7|3 5 2|9 4 8|
|8 3 5|7 4 9|1 6 2|
|2 4 9|8 6 1|7 3 5|
+-----+-----+-----+


# computer scientist version

In [28]:
Ksci = np.array(
    [
        [9,0xF+1,0,0xC+1, 0,0,0,0,         0,0xA+1,0,0, 0,0,0,  7],
        [0,0,0,0xA+1,     0,0,0,0xF+1,     0,0,0,0xB+1, 8,5,0xD+1,0],
        [0xB+1,0,5,0,     0,0,0xD+1,7,     0,8,0,0,     1,0,6,0],
        [2,0,0,0,         0,0,0,1,         4,0,10,3,    0,0,0,0],
        
        [0,0,0,0,         0,2,0xF+1,0xD+1, 0,4,1,0,     0,0xE+1,8,5],
        [0,2,0,7,         0,0,0,0xC+1,     0,0xB+1,0,0, 0xA+1,0,4,0],     
        [0,0xC+1,0,0xD+1, 0,0,7,4,         0,6,0,0,     10,3,0,0],
        [10,0,4,5,        0xE+1,0,3,0,     0,0,7,0xD+1, 0,0,0,0],
        
        [0,0,1, 0,0,7, 0,0,6],
        [0,0,0, 3,0,0, 9,0,0],
        [0,3,0, 0,4,0, 0,6,0],
        [0,0,9, 0,0,1, 0,0,5],
    ]
)

<ipython-input-28-2f63f8fc78c3>:1: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray
  Ksci = np.array(


In [29]:
0xB

11

In [30]:
set(range(10))

{0, 1, 2, 3, 4, 5, 6, 7, 8, 9}

In [31]:
np.identity(3)

array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]])